# Write P11 annotations to the misconception datasets

This notebook combines the reformatted MathDial datasets with the valid P11 extraction cache. It updates `data/misconception/mathdial_train.csv` and `data/misconception/mathdial_test.csv` in place.

Each file is written atomically and reloaded for verification before the original file is replaced.

## 1. Setup

In [6]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

for root in [Path.cwd(), *Path.cwd().parents]:
    if (root / 'extension').is_dir() and (root / 'data').is_dir():
        os.chdir(root)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the repository.')

sys.path.insert(0, str(Path.cwd()))

from extension.scripts.data_management.apply_cached_annotations import (
    apply_available_cache,
    load_available_cache,
    save_in_place,
    split_summary,
)
from extension.scripts.data_management.load_annotation_data import load_dataset

MODEL_SLUG = 'moonshot-direct/kimi-k3-max'
MODEL_DIR = MODEL_SLUG.replace('/', '__').replace(':', '-')
PROMPT = 'P11'
CACHE_ROOT = Path('extension/artifacts/extraction_cache')
DATA_DIR = Path('data/misconception')

SPLITS = {
    'train': 'mathdial_train.csv',
    'test': 'mathdial_test.csv',
}

print('Dataset directory:', DATA_DIR)

Dataset directory: data/misconception


## 2. Cache-to-dataset helper

For each split, the helper loads the misconception CSV, accepts only valid cache records, checks their dialogue and turn structure, applies the available annotations, and verifies the CSV after atomically overwriting it. Invalid or mismatched cache is reported and skipped.

In [7]:
def update_annotated_split(split):
    filename = SPLITS[split]
    dataset_path = DATA_DIR / filename
    cache_dir = CACHE_ROOT / split / MODEL_DIR / PROMPT

    source = load_dataset(dataset_path)
    cache, cache_audit, suffixed = load_available_cache(
        cache_dir,
        split=split,
        model_slug=MODEL_SLUG,
        prompt=PROMPT,
    )
    updated, apply_audit = apply_available_cache(source, cache)
    write_result = save_in_place(updated, dataset_path)

    report = split_summary(source, updated, apply_audit)
    report['cache files found'] = len(cache_audit)
    report['usable cache files'] = len(cache)
    report['ignored suffixed files'] = len(suffixed)
    report['output size (MiB)'] = write_result['size_mib']
    display(report.to_frame(split))
    print('Written:', write_result['path'])

    cache_issues = cache_audit.loc[cache_audit['status'].ne('usable')]
    apply_issues = apply_audit.loc[apply_audit['status'].ne('applied')]
    if not cache_issues.empty:
        print('Cache records skipped:')
        display(cache_issues)
    if not apply_issues.empty:
        print('Structurally mismatched annotations skipped:')
        display(apply_issues)

    return updated, report

## 3. Update the train dataset

In [8]:
train_annotated, train_report = update_annotated_split('train')

,train
dataset dialogues,2253.000000
usable cache records considered,2253.000000
cache records applied,2253.000000
structurally skipped records,0.000000
dataset dialogues not updated this run,0.000000
nonblank P/A/N cells after update,78045.000000
cache files found,2253.000000
usable cache files,2253.000000
ignored suffixed files,0.000000
output size (MiB),73.678431


Written: data/misconception/mathdial_train.csv


## 4. Update the test dataset

In [9]:
test_annotated, test_report = update_annotated_split('test')

,test
dataset dialogues,595.000000
usable cache records considered,595.000000
cache records applied,595.000000
structurally skipped records,0.000000
dataset dialogues not updated this run,0.000000
nonblank P/A/N cells after update,19510.000000
cache files found,595.000000
usable cache files,595.000000
ignored suffixed files,0.000000
output size (MiB),18.284484


Written: data/misconception/mathdial_test.csv


## 5. Write summary

The summary confirms how much valid cache was applied to each final misconception dataset.

In [10]:
review = pd.concat({'train': train_report, 'test': test_report}, axis=1).T
display(review)

,dataset dialogues,usable cache records considered,cache records applied,structurally skipped records,dataset dialogues not updated this run,nonblank P/A/N cells after update,cache files found,usable cache files,ignored suffixed files,output size (MiB)
train,2253.0,2253.0,2253.0,0.0,0.0,78045.0,2253.0,2253.0,0.0,73.678431
test,595.0,595.0,595.0,0.0,0.0,19510.0,595.0,595.0,0.0,18.284484
